In [0]:
from pyspark.sql.functions import *
from dateutil import parser

# ---------------- CONFIG ----------------
CATALOG = "chatbot_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Ensure silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

df = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.symptoms_data")

clean_date_format = (
    df.withColumn(
        "body_temperature_value",
        when(col("body_temperature_unit") == "F",
        round((col("body_temperature_value") - 32) * 5 / 9, 2)
    )
    .otherwise(col("body_temperature_value")))
    .drop("_rescued_data")
)

# Removes duplicates
clean_dedup = clean_date_format.dropDuplicates(["patient_id"])

(clean_dedup.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{CATALOG}.silver.symptoms_data")
)